In [9]:

    
import numpy as np
nsites = 8
nv = 2
sh = 6
trainfeatures = np.load("trainfeatures.npy")[sh::100, :nv, :].reshape(850, nsites).T
testfeatures = np.load("testfeatures.npy")[sh::100, :nv, :].reshape(120, nsites).T
traintargets = np.load("traintargets.npy")[sh::100, :nv, :].reshape(850, nsites).T
testtargets = np.load("testtargets.npy")[sh::100, :nv, :].reshape(120, nsites).T

#"pauli_prop_values.h5", "w") do file
#    write(file, "qrc_train", expectations)
#    write(file, "qrc_test", expectations_test)
#    write(file, "test_targets", testtargets)
#    write(file, "train_targets", traintargets)
#    write(file, "test_features", testfeatures)
#    write(file, "train_features", trainfeatures)

In [10]:
def sample_z0z1(exact_expectation: float, n_shots: int) -> float:
        """
        Simulate finite sampling of the Z0Z1 operator expectation value.

        Args:
            exact_expectation: The exact expectation value of Z0Z1, must be in [-1, 1]
            n_shots: Number of measurement shots

        Returns:
            Approximate expectation value from finite sampling
        """
        if not -1 <= exact_expectation <= 1:
            raise ValueError(f"Expectation value must be in [-1, 1], got {exact_expectation}")
        if n_shots <= 0:
            raise ValueError(f"n_shots must be positive, got {n_shots}")

        p_plus1 = (1 + exact_expectation) / 2
        k = np.random.binomial(n_shots, p_plus1)

        return (2 * k - n_shots) / n_shots

In [11]:
import numpy as np
statsfeatures = np.zeros((970, nsites))
allpredict = np.zeros((970, nsites))
statsfeatures[:850,:] = trainfeatures.T
statsfeatures[850:,:] = testfeatures.T
allpredict[:850,:] = traintargets.T
allpredict[850:,:] = testtargets.T

In [12]:

#test_embeddingsn = np.load("test_embeddings_noiselessl.npy")
#print(test_embeddingsn[:,2])
#print(test_embeddings[:,2])


In [13]:

class CRC_Evolve():
    def __init__(self, alpha, rho, rho2, resSize):
        self.alpha = alpha
        self.rho = rho
        self.rho2 = rho2
        self.resSize = resSize

        self.init_crc()
        from sklearn.linear_model import Ridge
        self.ridge = Ridge(alpha=1.0, tol=1.e-4, max_iter=8000) # You can adjust alpha

    def init_crc(self):
        """Initialize the Classical Reservoir"""
        # generate the ESN reservoir
        self.inSize = 8
        self.outSize = 1
        self.resSize = self.resSize
        np.random.seed(2)
        self.Win = (np.random.rand(self.resSize, self.inSize) - 0.5) * 1
        self.Win *= self.rho2

        self.W = np.random.rand(self.resSize, self.resSize) - 0.5 
        
        # normalizing and setting spectral radius (correct, slow):
        rhoW = max(abs(np.linalg.eig(self.W)[0]))
        self.W *= self.rho / rhoW
        self.init_xinit = np.tanh(np.ones((self.resSize))/np.sqrt(self.resSize))
    
    def reinit_x(self):
        self.x = self.init_xinit.copy()

    
    def evolve_crc(self, u):
        """Evolve the classical reservoir given the previous time t0"""
        xlist = []
        self.reinit_x()
        for c in range(10):
            
            self.x = ((1-self.alpha)*self.x
                    + self.alpha*(np.tanh( np.dot( self.Win, u) + np.dot( self.W, self.x ) )))
            if c in [9]:
                xlist += [self.x.copy()]
        #print('after', np.array(xlist).shape)
        return list(np.vstack(xlist).flatten())
    
    def generate_embeddings(self, train_features: np.ndarray, train_outputs: np.ndarray):
        """Evolve each set of time evolutions and train on results"""

        batch_size = train_features.shape[0]
        train_results = []
        for b in range(batch_size):
            self.reinit_x()
            train_results += [self.evolve_crc(train_features[b,:])]

        return train_results


    
    



In [14]:
crc = CRC_Evolve(0.5, 0.9, 0.8, 28)
classical_train = crc.generate_embeddings(statsfeatures[:850, :], allpredict[:850, :])
classical_test = crc.generate_embeddings(statsfeatures[850:, :], allpredict[850:, :])

In [15]:
#train_embeddings = np.load("train_embeddings_noiseless.npy")
#test_embeddings = np.load("test_embeddings_noiseless.npy")
train_embeddings = np.load("aquila_features.npy")
test_embeddings = np.load("aquila_test_features.npy")
#test_embeddingsn = np.load("test_embeddings_noiselessl.npy")
#print(test_embeddingsn[:,2])
#print(test_embeddings[:,2])
quantum_train = np.zeros((850,360))
quantum_test = np.zeros((120,360))

quantum_train = train_embeddings
quantum_test = test_embeddings[:,:]

In [16]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

t1 = 0
t2 = 0
all_r2 = np.zeros(12)
all_mse = np.zeros(12)
all_r2_qrc = np.zeros(12)
all_mse_qrc = np.zeros(12)
imps = []
ma = []
mb = []
n_windows = 1
bs = 3
ws = test_embeddings.shape[1]//10
for l in range(nsites):
    # Train simle linear
    X_Train = statsfeatures[t1:850]
    X_Test = statsfeatures[850+t2:]
    Y_Train = allpredict[t1:850,l] #np.concatenate((traintargets[:,i,:], traintargets[:,j,:], traintargets[:,k,:]), axis=1)[:,:]
    Y_Test = allpredict[850+t2:,l] #np.concatenate((testtargets[:,i,:], testtargets[:,j,:], testtargets[:,k,:]), axis=1)[:,:]
    alphas = np.logspace(-3, 4, 40)   # 100 candidates from 0.0001 to 10000
    from sklearn.linear_model import Ridge, RidgeCV
    ridge_cv = RidgeCV(
            alphas=alphas,
            cv=10,                          # 10-fold cross-validation
            scoring="neg_mean_squared_error"
)
    ridge_cv.fit(X_Train, Y_Train)
    alpha_best = ridge_cv.alpha_
    ridge_model = Ridge(alpha=alpha_best, tol=1.e-4, max_iter=8000) # You can adjust alpha
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_Train)
    X_test_scaled = scaler.transform(X_Test)
    ridge_model.fit(X_train_scaled, Y_Train)
    y_pred = ridge_model.predict(X_test_scaled)
    mse = mean_squared_error(Y_Test, y_pred)
    r2 = r2_score(Y_Test, y_pred)
    all_r2[l] = r2
    all_mse[l] = mse 
    alpha_best1 = alpha_best
    #print(X_Train.shape)
    
    # Train QRC
    #X_Train = np.concatenate((X_Train[:], classical_train, quantum_train[:850,ws*bs+8:ws*bs+ws*n_windows]), axis=1)#, quantum_train[:,-ws*n_windows:]), axis=1)
    #X_Test = np.concatenate((X_Test, classical_test, quantum_test[:,ws*bs+8:ws*bs+ws*n_windows]), axis=1)#, quantum_test[:,-ws*n_windows:]), axis=1)
    #X_Train = np.concatenate((X_Train[:], classical_train[t1:]), axis=1)#, quantum_train[:,-ws*n_windows:]), axis=1)
    #X_Test = np.concatenate((X_Test, classical_test[t2:]), axis=1)#, quantum_test[:,-ws*n_windows:]), axis=1)
    
    
    #X_Train = np.concatenate((X_Train[:], quantum_train[:850,ws*bs+8:ws*bs+ws*n_windows]), axis=1)#, quantum_train[:,-ws*n_windows:]), axis=1)
    #X_Test = np.concatenate((X_Test, quantum_test[:,ws*bs+8:ws*bs+ws*n_windows]), axis=1)#, quantum_test[:,-ws*n_windows:]), axis=1)
    
    X_Train = np.concatenate((X_Train[:], classical_train[t1:], quantum_train[t1:850,8:]), axis=1)#, quantum_train[:,-ws*n_windows:]), axis=1)
    X_Test = np.concatenate((X_Test, classical_test[t2:], quantum_test[t2:,8:]), axis=1)
    #X_Train = np.concatenate((X_Train[:], quantum_train[t1:850,8:]), axis=1)#, quantum_train[:,-ws*n_windows:]), axis=1)
    #X_Test = np.concatenate((X_Test, quantum_test[t2:,8:]), axis=1)
    ridge_cv = RidgeCV(
            alphas=alphas,
            cv=10,                          # 10-fold cross-validation
            scoring="neg_mean_squared_error"
    )
    
    ridge_cv.fit(X_Train, Y_Train)
    alpha_best = ridge_cv.alpha_
    #print(alpha_best1, alpha_best)
    #print(alpha_best)
    ridge_model = Ridge(alpha=alpha_best, tol=1.e-4, max_iter=8000) # You can adjust alpha
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_Train)
    X_test_scaled = scaler.transform(X_Test)
    ridge_model.fit(X_train_scaled, Y_Train[:])
    y_pred = ridge_model.predict(X_test_scaled)
    mse = mean_squared_error(Y_Test, y_pred)
    r2 = r2_score(Y_Test, y_pred)
    all_r2_qrc[l] = r2
    all_mse_qrc[l] = mse 
    if all_r2[l] > -0.5 and ((l < 12) or (l > 35 and l < (36+12))):
        
        print(f"{l} {all_r2[l]:4.3}    &      {all_r2_qrc[l]:4.3}   &    {all_r2_qrc[l]-all_r2[l]:4.3} \\\\")
        imps += [all_r2[l]-all_r2_qrc[l]]
        ma += [all_r2[l]]
        mb += [all_r2_qrc[l]]

0 0.795    &      0.79   &    -0.0046 \\
1 0.443    &      0.468   &    0.0247 \\
2 0.333    &      0.302   &    -0.0309 \\
3 -0.0507    &      -0.0253   &    0.0254 \\
4 -0.12    &      0.0361   &    0.156 \\
5 0.837    &      0.827   &    -0.0105 \\
6 0.0228    &      0.013   &    -0.00971 \\
7 0.125    &      0.248   &    0.123 \\


In [17]:
import numpy as np


class NeuralNetwork:
    """
    A 4-layer neural network for regression.

    Architecture:
        Input Layer  : 8 features
        Hidden Layer 1: 64 neurons  (ReLU)
        Hidden Layer 2: 32 neurons  (ReLU)
        Hidden Layer 3: 16 neurons  (ReLU)
        Output Layer :  1 neuron    (Linear)
    """

    def __init__(self, input_size=8, lr=0.001):
        self.lr = lr
        self.layers = [input_size, 64, 32, 16, 1]

        # He initialisation (good default for ReLU networks)
        self.W = [
            np.random.randn(self.layers[i], self.layers[i + 1])
            * np.sqrt(2.0 / self.layers[i])
            for i in range(len(self.layers) - 1)
        ]
        self.b = [np.zeros((1, self.layers[i + 1]))
                  for i in range(len(self.layers) - 1)]

    # ------------------------------------------------------------------
    # Activations
    # ------------------------------------------------------------------

    def _relu(self, x):
        return np.maximum(0, x)

    def _relu_deriv(self, x):
        return (x > 0).astype(float)

    # ------------------------------------------------------------------
    # Forward pass
    # ------------------------------------------------------------------

    def forward(self, X):
        """Return predictions and cache intermediate values for backprop."""
        self.cache = {"A0": X}
        A = X

        for i in range(len(self.W) - 1):          # hidden layers (ReLU)
            Z = A @ self.W[i] + self.b[i]
            A = self._relu(Z)
            self.cache[f"Z{i+1}"] = Z
            self.cache[f"A{i+1}"] = A

        # Output layer — linear activation for regression
        Z_out = A @ self.W[-1] + self.b[-1]
        self.cache[f"Z{len(self.W)}"] = Z_out
        self.cache[f"A{len(self.W)}"] = Z_out          # linear → A == Z
        return Z_out

    # ------------------------------------------------------------------
    # Loss  (Mean Squared Error)
    # ------------------------------------------------------------------

    def loss(self, y_pred, y_true):
        return np.mean((y_pred - y_true) ** 2)

    # ------------------------------------------------------------------
    # Backward pass
    # ------------------------------------------------------------------

    def backward(self, y_true):
        m = y_true.shape[0]
        n_layers = len(self.W)

        grads_W = [None] * n_layers
        grads_b = [None] * n_layers

        # Gradient of MSE w.r.t. output
        dA = 2 * (self.cache[f"A{n_layers}"] - y_true) / m

        # Backpropagate through each layer (output → input)
        for i in reversed(range(n_layers)):
            Z = self.cache[f"Z{i+1}"]
            A_prev = self.cache[f"A{i}"]

            # For hidden layers apply ReLU derivative; output layer is linear
            dZ = dA if i == n_layers - 1 else dA * self._relu_deriv(Z)

            grads_W[i] = A_prev.T @ dZ
            grads_b[i] = np.sum(dZ, axis=0, keepdims=True)
            dA = dZ @ self.W[i].T

        # Gradient descent update
        for i in range(n_layers):
            self.W[i] -= self.lr * grads_W[i]
            self.b[i] -= self.lr * grads_b[i]

    # ------------------------------------------------------------------
    # Training loop
    # ------------------------------------------------------------------

    def train(self, X, y, epochs=1000, verbose=True):
        for epoch in range(1, epochs + 1):
            y_pred = self.forward(X)
            current_loss = self.loss(y_pred, y)
            self.backward(y)

            if verbose and epoch % 100 == 0:
                print(f"Epoch {epoch:>5}/{epochs}  |  MSE Loss: {current_loss:.6f}")

    def predict(self, X):
        return self.forward(X)



    

In [18]:
np.random.seed(42)

# --- Generate synthetic data ---
N_SAMPLES = 500
X = np.random.randn(N_SAMPLES, 8)           # 8 input features
# True relationship: weighted sum of features + noise
true_weights = np.array([1.5, -2.0, 0.8, 3.1, -1.2, 0.5, 2.3, -0.9])
y = (X @ true_weights + np.random.randn(N_SAMPLES) * 0.5).reshape(-1, 1)

l=7
# --- Simple train / test split (80 / 20) ---
X_train = statsfeatures[:850,:]
X_test = statsfeatures[850:,:]
y_train = allpredict[:850,l:l+1] #np.concatenate((traintargets[:,i,:], traintargets[:,j,:], traintargets[:,k,:]), axis=1)[:,:]
y_test = allpredict[850:,l:l+1]
#split = int(0.8 * N_SAMPLES)
#X_train, X_test = X[:split], X[split:]
#y_train, y_test = y[:split], y[split:]

# --- Build and train the network ---
nn = NeuralNetwork(input_size=8, lr=0.4)

print("Training…\n")
nn.train(X_train, y_train, epochs=20000, verbose=True)

# --- Evaluate ---
y_pred = nn.predict(X_test)
test_mse = nn.loss(y_pred, y_test)
test_rmse = np.sqrt(test_mse)
print(f"\nTest MSE : {test_mse:.6f}")
print(f"Test RMSE: {test_rmse:.6f}")
print(f"Test R2 = {r2_score(y_test, y_pred)}")

Training…

Epoch   100/20000  |  MSE Loss: 0.000571
Epoch   200/20000  |  MSE Loss: 0.000468
Epoch   300/20000  |  MSE Loss: 0.000400
Epoch   400/20000  |  MSE Loss: 0.000358
Epoch   500/20000  |  MSE Loss: 0.000337
Epoch   600/20000  |  MSE Loss: 0.000315
Epoch   700/20000  |  MSE Loss: 0.000301
Epoch   800/20000  |  MSE Loss: 0.000293
Epoch   900/20000  |  MSE Loss: 0.000287
Epoch  1000/20000  |  MSE Loss: 0.000281
Epoch  1100/20000  |  MSE Loss: 0.000276
Epoch  1200/20000  |  MSE Loss: 0.000272
Epoch  1300/20000  |  MSE Loss: 0.000268
Epoch  1400/20000  |  MSE Loss: 0.000264
Epoch  1500/20000  |  MSE Loss: 0.000261
Epoch  1600/20000  |  MSE Loss: 0.000258
Epoch  1700/20000  |  MSE Loss: 0.000256
Epoch  1800/20000  |  MSE Loss: 0.000253
Epoch  1900/20000  |  MSE Loss: 0.000251
Epoch  2000/20000  |  MSE Loss: 0.000249
Epoch  2100/20000  |  MSE Loss: 0.000246
Epoch  2200/20000  |  MSE Loss: 0.000244
Epoch  2300/20000  |  MSE Loss: 0.000242
Epoch  2400/20000  |  MSE Loss: 0.000240
Epoch